> ⚠️ **시작하기 전에 — 클러스터를 연결하세요**
>
> 이 실습에는 **Photon이 비활성화된 클래식 컴퓨트 클러스터**가 필요합니다. 서버리스 컴퓨트는 실습에서 사용하는 Spark 구성 옵션을 지원하지 않으며, Photon은 이 실습이 노출하도록 설계된 skew 및 shuffle 문제를 자동으로 최적화하여 제거합니다 — 둘 중 하나라도 활성화되어 있으면 Spark UI에서 비정상적인 것을 전혀 볼 수 없습니다.
>
> 1. 이 노트북 상단의 컴퓨트 선택기에서 **Connect**를 클릭합니다.
> 2. **`perf-lab`** 클러스터를 선택합니다(실습 설정 지침에서 생성됨 — 비-Photon 런타임, Spark 구성에 `spark.databricks.photon.enabled false` 설정).
> 3. 셀을 실행하기 전에 클러스터 상태가 **Running**으로 표시될 때까지 기다립니다.

## 연습 1: 환경 설정

성능 문제를 조사하기 전에 작업할 데이터가 필요합니다. 이 연습에서는 Unity Catalog 구조를 만들고 두 개의 Delta 테이블을 생성합니다:

- **`perf_lab.data.transactions`** — 의도적으로 skew된 `customer_id` 컬럼을 가진 500,000개 행(~70%의 행이 단일 고객에 속함)
- **`perf_lab.data.customers`** — 200개 행(작은 차원 테이블)

계속하기 전에 이 연습의 모든 셀을 위에서 아래로 실행하세요.

### 작업 1.1: Unity Catalog 구조 만들기

이 셀은 제공됩니다 — 실행하여 카탈로그와 스키마를 만드세요.

In [ ]:
# 기본 스토리지가 활성화되어 있지 않으므로, 기본 카탈로그의 루트에서 스토리지 경로를 상속합니다.
catalogs = [catalog.name for catalog in spark.catalog.listCatalogs("adb_ws*")]
dplab13_default_catalog = catalogs[0] if catalogs else None

storage_root = (
    spark.sql(f"DESCRIBE CATALOG EXTENDED {dplab13_default_catalog}")
    .filter("info_name = 'Storage Root'")
    .select("info_value")
    .first()[0]
)
print (f"Storage root: {storage_root}")

In [ ]:
# ✅ 이 셀을 실행하세요 — 카탈로그와 스키마를 만듭니다
spark.sql(f"CREATE CATALOG IF NOT EXISTS perf_lab MANAGED LOCATION '{storage_root}' COMMENT '성능 문제 해결 실습'")
spark.sql("CREATE SCHEMA IF NOT EXISTS perf_lab.data COMMENT '성능 연습용 트랜잭션 데이터'")
print("✅ 카탈로그와 스키마가 준비되었습니다.")

### 작업 1.2: transactions 테이블 생성

데이터셋은 전적으로 Spark에서 생성됩니다 — 외부 파일이 필요하지 않습니다.

**의도적 skew:** `rand(seed=42) < 0.70`은 전체 행의 약 70%를 `customer_001`로 라우팅합니다. 나머지 모든 행은 `customer_002`부터 `customer_200`까지 분산됩니다. 연습 2에서 이 불균형의 결과를 보게 됩니다.

In [ ]:
# ✅ 이 셀을 실행하세요 — 심하게 skew된 customer_id로 500,000개의 트랜잭션을 생성합니다
from pyspark.sql import functions as F

n_rows = 500_000

df_transactions = (
    spark.range(n_rows)
    .withColumn(
        "customer_id",
        F.when(F.rand(seed=42) < 0.70, F.lit("customer_001"))
         .otherwise(
             F.concat(
                 F.lit("customer_"),
                 F.lpad((F.floor(F.rand(seed=99) * 199) + 2).cast("string"), 3, "0")
             )
         )
    )
    .withColumn("amount",   F.round(F.rand(seed=7) * 999 + 1, 2))
    .withColumn("txn_type", F.when(F.rand(seed=3) < 0.5, F.lit("purchase")).otherwise(F.lit("refund")))
    .withColumn("txn_id",   F.concat(F.lit("txn_"), F.col("id").cast("string")))
    .drop("id")
)

df_transactions.write.format("delta").mode("overwrite").saveAsTable("perf_lab.data.transactions")
count = spark.table("perf_lab.data.transactions").count()
print(f"✅ perf_lab.data.transactions에 {count:,}개 행을 작성했습니다")

### 작업 1.3: customers 테이블 생성

이 작은 차원 테이블은 200개 행을 가집니다 — 고객당 한 행입니다. 단일 executor의 메모리에 충분히 들어갈 만큼 작습니다. 이 사실을 기억해 두세요. 연습 3에서 중요합니다.

In [ ]:
# ✅ 이 셀을 실행하세요 — 200개의 customer 차원 행을 생성합니다
customer_rows = [
    (f"customer_{i:03d}", f"Customer {i}", f"region_{(i % 5) + 1}")
    for i in range(1, 201)
]

df_customers = spark.createDataFrame(customer_rows, ["customer_id", "name", "region"])
df_customers.write.format("delta").mode("overwrite").saveAsTable("perf_lab.data.customers")
print(f"✅ perf_lab.data.customers에 {df_customers.count()}개 행을 작성했습니다")

### 작업 1.4: skew 확인

아래 셀을 실행하여 데이터가 심하게 불균형한지 확인하세요. 출력은 트랜잭션 수 기준 상위 고객과 전체 데이터셋에서 차지하는 비율을 보여줍니다.

In [ ]:
# ✅ 이 셀을 실행하세요 — customer_id 값 전반의 행 분포를 보여줍니다
df = spark.table("perf_lab.data.transactions")
total = df.count()

top_customers = (
    df.groupBy("customer_id")
      .agg(F.count("*").alias("txn_count"))
      .withColumn("pct_of_total", F.round(F.col("txn_count") / total * 100, 1))
      .orderBy(F.desc("txn_count"))
      .limit(5)
)

print(f"Total rows: {total:,}")
print("Top 5 customers by transaction count:")
top_customers.show()

---

## 연습 2: 데이터 skew 노출

Spark UI에서 skew를 명확하게 보이게 하려면 **Adaptive Query Execution (AQE)**를 꺼야 합니다. AQE가 활성화되어 있으면 Databricks가 런타임에 과도하게 큰 파티션을 자동으로 분할하여 skew를 가리지만 UI에서 문제도 숨깁니다.

`customer_id`로 shuffle하는 두 가지 작업을 실행합니다:
1. **groupBy 집계** — shuffle이 ~70%의 행을 단일 reducer 작업으로 라우팅합니다
2. transactions와 customers 간의 **sort-merge join** — auto-broadcast가 비활성화되어 있어 Spark가 양쪽을 모두 shuffle해야 합니다

아래 셀을 실행한 후 **실습 지침 페이지로 돌아가서** **Part 2**의 단계를 따라 Spark UI에서 결과를 조사하세요.

### 작업 2.1: AQE 비활성화 및 shuffle 파티션 설정

In [ ]:
# ✅ 이 셀을 실행하세요 — AQE와 auto-broadcast를 비활성화하여 Spark UI에서 skew가 완전히 보이도록 합니다
spark.conf.set("spark.databricks.optimizer.adaptive.enabled", "false")
spark.conf.set("spark.sql.adaptive.enabled",                 "false")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold",       "-1")    # sort-merge join 강제
spark.conf.set("spark.sql.shuffle.partitions",               "50")

print("AQE disabled | auto-broadcast disabled | shuffle partitions: 50")

### 작업 2.2: skew된 집계 실행

`groupBy("customer_id")`는 shuffle을 트리거합니다. `customer_001`이 ~350,000개 행을 가지고 있기 때문에 하나의 파티션(따라서 하나의 작업)이 전체 데이터의 약 70%를 받습니다. 다른 모든 파티션은 나머지 30%를 나눠 가집니다.

In [ ]:
# ✅ 이 셀을 실행하세요 — customer_id에 대한 skew된 shuffle을 트리거합니다
import time

df_tx = spark.table("perf_lab.data.transactions")

start = time.time()
result_agg = (
    df_tx
    .groupBy("customer_id", "txn_type")
    .agg(
        F.sum("amount").alias("total_amount"),
        F.count("*").alias("txn_count"),
        F.avg("amount").alias("avg_amount")
    )
)
result_agg.write.format("delta").mode("overwrite").saveAsTable("perf_lab.data.agg_skewed")
elapsed = time.time() - start
print(f"✅ Aggregation complete in {elapsed:.1f}s — {spark.table('perf_lab.data.agg_skewed').count()} rows")

### 작업 2.3: skew된 sort-merge join 실행

Auto-broadcast가 비활성화되어 있으므로(`autoBroadcastJoinThreshold = -1`) Spark는 join의 **양쪽**을 모두 shuffle하고 sort-merge 전략을 사용해야 합니다. skew된 `customer_id` 분포로 인해 하나의 reducer 작업이 transactions 쪽에서 불균형하게 큰 파티션을 받습니다.

In [ ]:
# ✅ 이 셀을 실행하세요 — skew된 sort-merge join을 트리거합니다
df_tx   = spark.table("perf_lab.data.transactions")
df_cust = spark.table("perf_lab.data.customers")

start = time.time()
result_join = (
    df_tx
    .join(df_cust, "customer_id", "left")
    .groupBy("region", "txn_type")
    .agg(
        F.sum("amount").alias("total_amount"),
        F.count("*").alias("txn_count")
    )
)
result_join.write.format("delta").mode("overwrite").saveAsTable("perf_lab.data.join_skewed")
elapsed = time.time() - start
print(f"✅ Sort-merge join complete in {elapsed:.1f}s — {spark.table('perf_lab.data.join_skewed').count()} rows")

> **이제 실습 지침 페이지로 돌아가서 Part 2의 단계를 따라 Spark UI를 조사하세요.**
>
> 조사를 마쳤으면 아래 연습 3을 계속하세요.

---

## 연습 3: 데이터 skew 수정

skew를 해결하기 위한 두 가지 상호 보완적인 도구가 있습니다:

| 기법 | 작동 방식 | 가장 적합한 경우 |
|-----------|-------------|----------|
| **Broadcast join** | 작은 테이블을 모든 executor로 전송 — 양쪽 모두 shuffle하지 않음 | join의 한쪽이 작을 때 |
| **AQE skew join** | 과도하게 큰 shuffle 파티션을 런타임에 동적으로 분할 | join 및 집계의 범용 skew |

이 연습에서는 두 가지를 모두 적용합니다.

### 🎯 작업 3.1: broadcast 힌트를 사용하여 join 수정

`customers` 테이블은 **200개 행**만 가집니다. 모든 executor의 메모리에 쉽게 들어갑니다. 이를 broadcast하면 `customers` 쪽의 shuffle이 완전히 제거되며, transactions 쪽의 `customer_id` skew가 더 이상 작업 불균형을 일으키지 않습니다.

**작업:** 작업 2.3의 join을 `customers` DataFrame에 대해 broadcast 힌트를 사용하도록 다시 작성하세요.

💡 **힌트:**
- PySpark 구문: 작은 DataFrame을 `F.broadcast(df)`로 감쌉니다
- SQL 힌트 구문: SQL 문자열 내부에 `/*+ BROADCAST(alias) */`
- broadcast 힌트는 Spark에게 sort-merge 전략을 건너뛰고 대신 작은 테이블을 모든 worker에 복사하도록 지시합니다

🤖 **Genie Code 프롬프트:**
> *"PySpark에서 작은 차원 테이블의 셔플링을 방지하기 위해 브로드캐스트 조인 힌트를 사용하는 방법은 무엇인가요?"*

In [ ]:
# 🎯 작업: 이 join을 df_cust에 대해 broadcast 힌트를 사용하도록 다시 작성하세요
# 경과 시간을 측정하고 작업 2.3의 sort-merge join과 비교하세요
import time
from pyspark.sql import functions as F

df_tx   = spark.table("perf_lab.data.transactions")
df_cust = spark.table("perf_lab.data.customers")

start = time.time()

# TODO: 아래의 일반 join을 broadcast join으로 교체하세요
result_broadcast = (
    df_tx
    .join(df_cust, "customer_id", "left")   # ← 여기에 broadcast 힌트를 적용하세요
    .groupBy("region", "txn_type")
    .agg(
        F.sum("amount").alias("total_amount"),
        F.count("*").alias("txn_count")
    )
)
result_broadcast.write.format("delta").mode("overwrite").saveAsTable("perf_lab.data.join_broadcast")
elapsed = time.time() - start
print(f"Join complete in {elapsed:.1f}s — {spark.table('perf_lab.data.join_broadcast').count()} rows")

### 작업 3.2: AQE를 다시 활성화하고 자동 skew 처리 관찰

`spark.sql.adaptive.skewJoin.enabled = true`이면 AQE가 shuffle 중에 skew된 파티션을 자동으로 감지하여 더 작은 하위 작업으로 분할합니다. 이는 코드 변경 없이 skew를 처리합니다 — 테이블을 broadcast할 수 없거나 원하지 않을 때 유용합니다.

아래 셀은 AQE를 다시 활성화하고 작업 2.3의 원래 sort-merge join을 다시 실행합니다(broadcast 힌트 없음). 실행 시간을 비교하고 Spark UI를 확인하여 AQE가 skew된 파티션을 어떻게 분할하는지 확인하세요.

In [ ]:
# ✅ 이 셀을 실행하세요 — AQE를 다시 활성화하고 sort-merge join을 다시 실행합니다
spark.conf.set("spark.databricks.optimizer.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.enabled",                 "true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled",        "true")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold",       "10485760")  # 10 MB로 재설정

print("AQE enabled | skewJoin enabled | auto-broadcast threshold: 10 MB")

df_tx   = spark.table("perf_lab.data.transactions")
df_cust = spark.table("perf_lab.data.customers")

start = time.time()
result_aqe = (
    df_tx
    .join(df_cust, "customer_id", "left")
    .groupBy("region", "txn_type")
    .agg(
        F.sum("amount").alias("total_amount"),
        F.count("*").alias("txn_count")
    )
)
result_aqe.write.format("delta").mode("overwrite").saveAsTable("perf_lab.data.join_aqe")
elapsed = time.time() - start
print(f"✅ AQE join complete in {elapsed:.1f}s — {spark.table('perf_lab.data.join_aqe').count()} rows")
print("\nCheck the Spark UI — AQE splits the skewed partition into multiple sub-tasks.")

---

## 연습 4: 과도한 shuffle 노출

Spark가 데이터를 재분배해야 할 때 shuffle은 불가피합니다 — 예를 들어 `groupBy`나 `join`의 경우입니다. 그러나 이미 shuffle을 포함하는 작업 전후에 `repartition()`을 호출하면 *불필요한* shuffle을 쉽게 도입할 수 있습니다.

이 연습의 파이프라인은 `groupBy` 다음에 `join`과 `sort`를 수행하지만, 각각 아무 이점 없이 전체 shuffle을 트리거하는 **세 개의 추가 `repartition()` 호출**을 추가합니다. Spark UI는 DAG에서 여러 개의 **Exchange** 노드를 보여줍니다 — 계산이 실제로 필요로 하는 것보다 더 많습니다.

아래 셀을 실행한 후 **실습 지침 페이지로 돌아가서** **Part 4**의 단계를 따라 shuffle 메트릭을 조사하세요.

### 작업 4.1: 구성 재설정 및 shuffle이 많은 파이프라인 실행

In [ ]:
# ✅ 이 셀을 실행하세요 — AQE와 auto-broadcast를 비활성화하여 모든 shuffle이 보이도록 합니다
spark.conf.set("spark.databricks.optimizer.adaptive.enabled", "false")
spark.conf.set("spark.sql.adaptive.enabled",                 "false")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold",       "-1")
spark.conf.set("spark.sql.shuffle.partitions",               "200")

print("AQE disabled | auto-broadcast disabled | shuffle partitions: 200")

In [ ]:
# ✅ 이 셀을 실행하세요 — 불필요한 repartition이 있는 shuffle이 많은 파이프라인
#
# Shuffle 1: repartition(100, "txn_type")     — groupBy 전 사전 repartition (불필요)
# Shuffle 2: groupBy aggregation              — customer_id에 대한 필요한 shuffle
# Shuffle 3: repartition(200, "customer_id")  — join 전 재-shuffle (불필요)
# Shuffle 4: sort-merge join with customers   — broadcast가 비활성화되어 shuffle
# Shuffle 5: repartition(50)                  — join 후 repartition (불필요)
# Shuffle 6: orderBy                          — 필요한 전역 정렬

df_tx   = spark.table("perf_lab.data.transactions")
df_cust = spark.table("perf_lab.data.customers")

start = time.time()
result_inefficient = (
    df_tx
    .repartition(100, "txn_type")                       # Shuffle 1: 불필요
    .groupBy("customer_id", "txn_type")
    .agg(
        F.sum("amount").alias("total_amount"),
        F.count("*").alias("txn_count"),
        F.avg("amount").alias("avg_amount")
    )
    .repartition(200, "customer_id")                    # Shuffle 3: 불필요
    .join(df_cust, "customer_id", "left")               # Shuffle 4: sort-merge (broadcast 비활성화)
    .repartition(50)                                    # Shuffle 5: 불필요
    .orderBy("region", "txn_type")                      # Shuffle 6: 필요
)
result_inefficient.write.format("delta").mode("overwrite").saveAsTable("perf_lab.data.results_inefficient")
elapsed = time.time() - start
print(f"✅ Inefficient pipeline complete in {elapsed:.1f}s")
print(f"   Rows written: {spark.table('perf_lab.data.results_inefficient').count():,}")
print("\nNow return to the lab instructions page and investigate the Spark UI for Part 4.")

> **실습 지침 페이지로 돌아가서 Part 4의 단계를 따라 shuffle 메트릭을 조사하세요.**
>
> 조사를 마쳤으면 아래 연습 5를 계속하세요.

---

## 연습 5: shuffle 오버헤드 줄이기

연습 4의 파이프라인은 `groupBy → join → sort`를 수행합니다. 그 계산은 근본적으로 **두 번**의 shuffle을 필요로 합니다: 하나는 집계용, 하나는 전역 정렬용입니다. 나머지는 모두 오버헤드입니다.

### 🎯 챌린지

shuffle 수를 최소화하도록 파이프라인을 다시 작성하세요. 최적화된 버전은 다음을 수행해야 합니다:

1. **모든 불필요한 `repartition()` 호출 제거** — `groupBy`, `join`, `orderBy`는 이미 데이터를 이동시킵니다. 사전 shuffle은 도움이 되지 않고 추가 네트워크 I/O만 더합니다.
2. sort-merge shuffle을 완전히 제거하기 위해 `customers` 테이블에 **broadcast join 사용**.
3. 실행 전에 **AQE를 다시 활성화**하여 Spark가 파티션 크기를 동적으로 추가 최적화할 수 있도록 합니다.
4. **`orderBy` 유지** — 출력은 여전히 정렬되어야 합니다.

경과 시간을 측정하고 연습 4의 실행 시간과 비교하세요.

💡 **힌트:**
- `repartition()` 호출은 매번 전체 shuffle을 트리거합니다 — 특정한 이유(예: 스토리지에 작성되는 출력 파일 수 제어)가 있을 때만 유지하세요.
- `customers`는 200개 행을 가집니다. `F.broadcast(df_cust)`를 사용하여 모든 executor에 복사하세요.
- broadcast join 후에는 `orderBy` shuffle만 남습니다 — 그것이 정렬된 결과를 위한 최소값입니다.

🤖 **Genie Code 프롬프트:**
> *"PySpark 파이프라인에 groupBy와 join 전에 여러 번의 repartition() 호출이 있습니다. 불필요한 repartition을 식별하고 셔플 횟수를 최소화하도록 파이프라인을 재작성하는 방법을 알려주세요. 또한 작은 차원 테이블에는 브로드캐스트 조인을 적용하는 방법도 알려주세요."*

In [ ]:
# 🎯 작업: 연습 4의 파이프라인을 shuffle을 최소화하도록 다시 작성하세요
#
# AQE와 auto-broadcast를 다시 활성화한 다음, 파이프라인의 최적화된 버전을 작성하세요.
# 시간을 측정하고 연습 4와 비교하세요.

# 1단계: AQE와 broadcast 임계값 다시 활성화
# TODO: spark.databricks.optimizer.adaptive.enabled와 spark.sql.adaptive.enabled를 true로 설정하세요
# TODO: spark.sql.autoBroadcastJoinThreshold를 10485760 (10 MB)으로 재설정하세요

df_tx   = spark.table("perf_lab.data.transactions")
df_cust = spark.table("perf_lab.data.customers")

start = time.time()

# 2단계: 최적화된 파이프라인 작성
# TODO: 불필요한 repartition을 제거하세요
# TODO: customers join에 F.broadcast()를 적용하세요
result_optimised = (
    df_tx
    # ... 여기에 코드를 작성하세요 ...
)
result_optimised.write.format("delta").mode("overwrite").saveAsTable("perf_lab.data.results_optimised")
elapsed = time.time() - start
print(f"Optimised pipeline complete in {elapsed:.1f}s")
print(f"Rows written: {spark.table('perf_lab.data.results_optimised').count():,}")
print("\nCompare in the Spark UI: fewer Exchange nodes, less Shuffle Read/Write per stage.")

---

## 정리 (선택 사항)

작업을 마쳤으면 아래 셀을 실행하여 Unity Catalog에서 모든 실습 리소스를 제거하세요.

In [ ]:
# 주석을 해제하고 실행하여 모든 실습 리소스를 제거하세요
# spark.sql("DROP CATALOG IF EXISTS perf_lab CASCADE")
# print("✅ Lab resources cleaned up")